In [2]:
from pynhd import NLDI
import geopandas as gpd
import pandas as pd
from supporting_scripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
from shapely.geometry import box, Polygon
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [3]:
station_id = "09010500" # NWIS id for Colorado River below Baker Gulch
basinname = 'ColoradoRiverBasin'

Copy process in data acquisition and analysis to get one dataframe which will be easier to work with


In [12]:
unprocessed_SNOTEL = {}
path = 'files/SNOTEL'

for filename in os.listdir(path):
    if filename.endswith('.csv'):
        name = filename.split('_')[1] 
        unprocessed_SNOTEL[name] = pd.read_csv(os.path.join(path, filename))
        
        unprocessed_SNOTEL[name]['datetime'] = pd.to_datetime(unprocessed_SNOTEL[name]['datetime'])
        unprocessed_SNOTEL[name].set_index('datetime', inplace=True)
        
        unprocessed_SNOTEL[name].rename(columns={'WTEQ': f"{name}_SWE_cm"}, inplace=True)
        unprocessed_SNOTEL[name][f"{name}_SWE_cm"] = unprocessed_SNOTEL[name][f"{name}_SWE_cm"] * 2.54
        
        # Keep only the SWE column
        unprocessed_SNOTEL[name] = unprocessed_SNOTEL[name][[f"{name}_SWE_cm"]]
        
        print(f"{name}: {len(unprocessed_SNOTEL[name])} start date: {unprocessed_SNOTEL[name].index.min()} end date: {unprocessed_SNOTEL[name].index.max()}")

565: 17347 start date: 1978-10-01 00:00:00 end date: 2026-03-29 00:00:00
688: 16616 start date: 1980-10-01 00:00:00 end date: 2026-03-29 00:00:00


In [14]:
#The site with the latest start date will guide the rest
latest_start_date = max([df.index.min() for df in unprocessed_SNOTEL.values()])

#The site with the earliest end date will guide the rest
soonest_end_date = min([df.index.max() for df in unprocessed_SNOTEL.values()])
for key in unprocessed_SNOTEL.keys():
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index >= latest_start_date]
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index <= soonest_end_date]

#merge all dictionary dataframes into one larger dataframe
SNOTEL_df = pd.concat(unprocessed_SNOTEL.values(), axis=1)
#set the date index to be the index of the first dataframe in the dictionary

SNOTEL_df.tail()

,565_SWE_cm,688_SWE_cm
datetime,,
2026-03-25,0.774192,NaN
2026-03-26,0.728980,0.161290
2026-03-27,0.709676,0.122682
2026-03-28,0.690372,0.077470
2026-03-29,0.677418,0.045212


Load Streamflow Data

In [15]:
streamflow_df = pd.read_csv(f"files/NWIS/streamflow_{station_id}.csv")
#set the date column to be a datetime object and set it to the index
streamflow_df['Date'] = pd.to_datetime(streamflow_df['Date'])
streamflow_df.set_index('Date', inplace=True)

streamflow_df.head()

,site_no,flow_cms
Date,,
1980-01-01,9010500,0.226534
1980-01-02,9010500,0.226534
1980-01-03,9010500,0.226534
1980-01-04,9010500,0.226534
1980-01-05,9010500,0.226534


Load Catchment Info

In [16]:
basin_info = pd.read_csv(f"files/basin_info/basin_info_{station_id}.csv")
basin_info.head()

,Unnamed: 0,station_id,Average_Elevation_m,Minimum_Elevation_m,Maximum_Elevation_m,Average_Slope,Area_km2,Open_Water,Perennial_Ice_Snow,"Developed,_Open_Space","Developed,_Low_Intensity","Developed,_Medium_Intensity","Developed,_High_Intensity",Barren_Land_Rock_Sand_Clay,Deciduous_Forest,Evergreen_Forest,Shrub_Scrub,Grassland_Herbaceous,Woody_Wetlands,Emergent_Herbaceous_Wetlands
0,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,292.895387,0.076363,3.148496,0.516917,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372


Merge into 1 Dataframe

In [ ]:
#find the latest start date and the earliest end date for SNOTEL_df cleaned
begin_date = max([df.index.min() for df in [SNOTEL_df,  streamflow_df]]) 
end_date = min([df.index.max() for df in [SNOTEL_df,  streamflow_df]]) 

#clip each dataframe to have the same begin and end dates
SNOTEL_df = SNOTEL_df[(SNOTEL_df.index >= begin_date) & (SNOTEL_df.index <= end_date)]

streamflow_df = streamflow_df[(streamflow_df.index >= begin_date) & (streamflow_df.index <= end_date)]


In [19]:
#merge the SNOTEL_df and streamflow dataframes
Hydro_df = pd.concat([SNOTEL_df, streamflow_df], axis=1)
#put the site_no column, second to last, and streamfow column, last column, as the first two columns in the dataframe
cols = Hydro_df.columns.tolist()
cols = cols[-2:] + cols[:-2]
Hydro_df = Hydro_df[cols]
Hydro_df.head()

,site_no,flow_cms,565_SWE_cm,688_SWE_cm
1980-10-01,9010500,0.453069,0.0,0.0
1980-10-02,9010500,0.396435,0.0,0.0
1980-10-03,9010500,0.368118,0.0,0.0
1980-10-04,9010500,0.368118,0.0,0.0
1980-10-05,9010500,0.368118,0.0,0.0


In [20]:
#all of the NaN values here should be 0, fill them
Hydro_df = Hydro_df.fillna(0)
Hydro_df.head()

,site_no,flow_cms,565_SWE_cm,688_SWE_cm
1980-10-01,9010500,0.453069,0.0,0.0
1980-10-02,9010500,0.396435,0.0,0.0
1980-10-03,9010500,0.368118,0.0,0.0
1980-10-04,9010500,0.368118,0.0,0.0
1980-10-05,9010500,0.368118,0.0,0.0


In [21]:
#add in the basin info as columns in the dataframe, repeat the values for each row
for col in basin_info.columns:
    Hydro_df[col] = basin_info[col][0]

Hydro_df.head()

,site_no,flow_cms,565_SWE_cm,688_SWE_cm,Unnamed: 0,station_id,Average_Elevation_m,Minimum_Elevation_m,Maximum_Elevation_m,Average_Slope,...,"Developed,_Low_Intensity","Developed,_Medium_Intensity","Developed,_High_Intensity",Barren_Land_Rock_Sand_Clay,Deciduous_Forest,Evergreen_Forest,Shrub_Scrub,Grassland_Herbaceous,Woody_Wetlands,Emergent_Herbaceous_Wetlands
1980-10-01,9010500,0.453069,0.0,0.0,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
1980-10-02,9010500,0.396435,0.0,0.0,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
1980-10-03,9010500,0.368118,0.0,0.0,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
1980-10-04,9010500,0.368118,0.0,0.0,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
1980-10-05,9010500,0.368118,0.0,0.0,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372


In [22]:
#take a look around peak SWE to make sure we have snotel values, early season can be tricky to assess
Hydro_df.loc['2019-03-01':'2019-04-01']

,site_no,flow_cms,565_SWE_cm,688_SWE_cm,Unnamed: 0,station_id,Average_Elevation_m,Minimum_Elevation_m,Maximum_Elevation_m,Average_Slope,...,"Developed,_Low_Intensity","Developed,_Medium_Intensity","Developed,_High_Intensity",Barren_Land_Rock_Sand_Clay,Deciduous_Forest,Evergreen_Forest,Shrub_Scrub,Grassland_Herbaceous,Woody_Wetlands,Emergent_Herbaceous_Wetlands
2019-03-01,9010500,0.219738,1.193546,0.470916,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
2019-03-02,9010500,0.217190,1.219454,0.490220,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
2019-03-03,9010500,0.213226,1.271016,0.516128,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
2019-03-04,9010500,0.211526,1.283970,0.509778,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
2019-03-05,9010500,0.219455,1.283970,0.503174,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
2019-03-06,9010500,0.230499,1.296670,0.516128,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
2019-03-07,9010500,0.238427,1.367790,0.580644,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
2019-03-08,9010500,0.241259,1.387094,0.586994,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
2019-03-09,9010500,0.239560,1.425702,0.612902,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
2019-03-10,9010500,0.234180,1.438656,0.619252,ColoradoRiverBasin,9010500,3200.7634,2659.6445,3941.0696,0.363304,...,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372


Task 2: 
 SWE analysis for SNOTEL site(s)

- A description of the SNOTEL sites(s) (e.g, location, elevation, length of operations).
- Figures or tables with captions that communicate the historical range of SWE, similar to a SNOTEL site figure.
- Discuss how the SWE compares to the historical mean/median for April 1st, 2025.

In [8]:
#load snotel data
unprocessed_SNOTEL = {}
#read all files in the following path into the dictionary
path = 'files/SNOTEL'
for filename in os.listdir(path):
    if filename.endswith('.csv'):
        #select the name of the file between the _ and _
        name = filename.split('_')[1] 
        unprocessed_SNOTEL[name] = pd.read_csv(os.path.join(path, filename))
        #make the date a datetime object and set to the index
        unprocessed_SNOTEL[name]['Date'] = pd.to_datetime(unprocessed_SNOTEL[name]['Date'])
        unprocessed_SNOTEL[name].set_index('Date', inplace=True)
        #rename the Snow Water Equivalent (m) Start of Day Values to SWE_cm
        unprocessed_SNOTEL[name].rename(columns={'Snow Water Equivalent (m) Start of Day Values': f"{name}_SWE_cm"}, inplace=True)
        #convert SWE_m to cm
        unprocessed_SNOTEL[name][f"{name}_SWE_cm"] = unprocessed_SNOTEL[name][f"{name}_SWE_cm"] * 100
        #remove the Water_Year column
        unprocessed_SNOTEL[name].drop(columns=['Water_Year'], inplace=True)
        #we need to know how many obs for each DF, print the df name, its length, and the start/end dates
        print(f"{name}: {len(unprocessed_SNOTEL[name])} start date: {unprocessed_SNOTEL[name].index.min()} end date: {unprocessed_SNOTEL[name].index.max()}")
    

KeyError: 'Date'